# GameTheory 3e : Meta-Actions Tarifees -- l'agent qui joue, puis qui change les regles

[← GameTheory-03-Topology2x2](GameTheory-03-Topology2x2.ipynb) | [GameTheory-03b-Chambres-et-Murs →](GameTheory-03b-Chambres-et-Murs.ipynb) | [↑ README GameTheory](README.md)

**Versant D4 du chantier #12207.** Les notebooks [GT-3](GameTheory-03-Topology2x2.ipynb), [GT-3b](GameTheory-03b-Chambres-et-Murs.ipynb) et [GT-3h](GameTheory-03h-Deux-Especes-de-Fleches.ipynb) ont fait de l'espace des jeux 2x2 ordinaux un **univers fini manipulable** : 576 chambres strictes, six swaps generateurs, des murs ou vivent les egalites. Ce notebook monte l'echelle d'un cran : les swaps y deviennent des **actions que l'agent paie**.

## Plan

1. **Marche 1 -- l'agent joue** : equilibres de Nash purs et dynamique de meilleure reponse sur les 576 jeux
2. **Marche 2 -- l'agent deplace le jeu** : meta-actions tarifees, un cout mesure en echelons de rang, seuil de migration
3. **Marche 3 -- deux agents en desaccord** : le meta-jeu 4x4, ses equilibres, et la question de l'accord

## La question

La strate 6 de la theorie des jeux dit ce qu'un agent fait *dans* des regles donnees. La strate 7 demande ce qui se passe quand **modifier les regles est lui-meme une action** -- avec un cout, donc un arbitrage, donc un equilibre au niveau superieur. Rendu operationnel sur un univers fini, le franchissement devient mesurable : *a quel prix un agent accepte-t-il de deplacer son propre jeu, et que devient cet arbitrage quand les deux joueurs le jouent simultanement ?*

## 0. Le substrat, herite de GT-3b

Un jeu = **deux tables de rangs**, une par joueur, chacune un 4-uplet ordonnant strictement les quatre cases (haut-gauche, haut-droite, bas-gauche, bas-droite) -- rang 4 = meilleur. Les **swaps** `R1, R2, R3` (cote Ligne) et `C1, C2, C3` (cote Colonne) echangent les cases portant deux rangs adjacents `k` et `k+1` : ce sont les six generateurs de l'espace, chacun traverse un mur du monde de Bruns-Kimmich. Toute la mecanique est reprise tel quel de [GT-3b](GameTheory-03b-Chambres-et-Murs.ipynb), sans modification -- ce notebook est un consommateur du substrat, pas une refonte.

In [1]:
# Substrat GT-3b, repris tel quel -- pur stdlib
from itertools import product
from collections import Counter, deque

def swap_val(t, k):
    """Echange les cases portant les rangs k et k+1 (traversee de la facette k<->k+1)."""
    pk, pk1 = t.index(k), t.index(k + 1)
    l = list(t); l[pk], l[pk1] = l[pk1], l[pk]; return tuple(l)

chambres = sorted({tuple(p) for p in product(range(1, 5), repeat=4) if len(set(p)) == 4})
jeux = [(r, c) for r in chambres for c in chambres]
print("Chambres strictes par joueur :", len(chambres), "| jeux 2x2 stricts :", len(jeux))
print("Generateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)")

# Le Dilemme du Prisonnier, encodage GT-21 / GT-3b
PD = ((3, 1, 4, 2), (3, 4, 1, 2))
print("PD : Ligne", PD[0], "Colonne", PD[1], "(rang 4 = meilleur, cases TG TD BG BD)")

Chambres strictes par joueur : 24 | jeux 2x2 stricts : 576
Generateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)
PD : Ligne (3, 1, 4, 2) Colonne (3, 4, 1, 2) (rang 4 = meilleur, cases TG TD BG BD)


## 1. Marche 1 -- l'agent joue

Avant de changer les regles, il faut les subir. Pour chaque jeu, deux instruments :

- les **equilibres de Nash purs** : cases ou personne ne prefere devier unilateralement ;
- la **dynamique de meilleure reponse** depuis la case haut-gauche : Ligne ajuste, puis Colonne, et on repete -- c'est la facon la plus simple dont des agents "jouent" quand personne ne calcule d'equilibre.

Sur des rangs *ordinaux*, l'equilibre mixte n'a pas de sens (on ne peut pas moyenner des rangs) : un jeu sans equilibre pur est ici **injouable** -- la dynamique cycle sans fin. C'est un fait, pas une anomalie a corriger.

In [2]:
# Marche 1 : NE purs + dynamique BR depuis la case haut-gauche
def ne_purs(row, col):
    nes = []
    for r in (0, 1):
        for c in (0, 1):
            if row[2*r+c] > row[2*(1-r)+c] and col[2*r+c] > col[2*r+(1-c)]:
                nes.append((r, c))
    return nes

def br_dyn(row, col, start=(0, 0), max_steps=16):
    """BR alterne (Ligne d'abord). ('cycle'|'ok', case finale)."""
    r, c = start
    for _ in range(max_steps):
        moved = False
        nr = r if row[2*r+c] > row[2*(1-r)+c] else (1 - r)
        if nr != r: r, moved = nr, True
        nc = c if col[2*r+c] > col[2*r+(1-c)] else (1 - c)
        if nc != c: c, moved = nc, True
        if not moved: return ('ok', (r, c))
    return ('cycle', (r, c))

dist_ne = Counter(len(ne_purs(r, c)) for r, c in jeux)
stats = Counter(br_dyn(r, c)[0] for r, c in jeux)
print("NE purs par jeu :", dict(sorted(dist_ne.items())), "sur", len(jeux))
print("Dynamique BR depuis TG : converge", stats['ok'], "| cycle", stats['cycle'])
croisement = Counter((len(ne_purs(r, c)), br_dyn(r, c)[0]) for r, c in jeux)
print("Croisement (#NE, BR) :", dict(sorted(croisement.items())))
print()
print("PD : NE purs =", ne_purs(*PD), "-> (bas,droite) = defection mutuelle, rang", PD[0][3], "chacun")

NE purs par jeu : {0: 72, 1: 432, 2: 72} sur 576
Dynamique BR depuis TG : converge 504 | cycle 72
Croisement (#NE, BR) : {(0, 'cycle'): 72, (1, 'ok'): 432, (2, 'ok'): 72}

PD : NE purs = [(1, 1)] -> (bas,droite) = defection mutuelle, rang 2 chacun


### Lecture de la marche 1

La distribution est remarquablement symetrique : **72 jeux sans equilibre pur, 432 avec un seul, 72 avec deux** (les coordinations). Et le croisement est exact : la dynamique converge **si et seulement si** un equilibre pur existe -- les 72 cycles sont precisement les 72 jeux sans equilibre. Aucun jeu a deux equilibres ne pose ici de probleme de selection *dynamique* : parti de haut-gauche, le jeu se stabilise toujours quelque part.

Le Dilemme du Prisonnier fait ce qu'on attend de lui : equilibre unique a la defection mutuelle, rang 2 chacun -- le rang 4 de la cooperation existe mais n'est pas stable. Retenons ce chiffre, **2** : c'est ce que vaut le Dilemme pour celui qui y est piege.

In [3]:
# Le fait brut : exhiber un jeu injouable (cycle BR complet)
ex_injouable = ((2, 3, 1, 4), (1, 2, 4, 3))
row, col = ex_injouable
print("Jeu sans NE pur : Ligne", row, "Colonne", col)
NOMS_CASES = ("TG", "TD", "BG", "BD")
r, c = 0, 0
trace = [(r, c)]
for _ in range(6):
    nr = r if row[2*r+c] > row[2*(1-r)+c] else (1 - r)
    if nr != r: r = nr; trace.append((r, c))
    nc = c if col[2*r+c] > col[2*r+(1-c)] else (1 - c)
    if nc != c: c = nc; trace.append((r, c))
print("Trajectoire BR depuis TG :", " -> ".join(NOMS_CASES[2*tr+tc] for tr, tc in trace[:6]) + " -> ... (cycle)")
print("Boucle :", " -> ".join(NOMS_CASES[2*tr+tc] for tr, tc in trace[1:5]) + " -> ... repetee indefiniment")

# Convention de la suite : un jeu injouable vaut 0 pour chacun (pire que tout rang jouable)
def payoff(row, col, cote):
    st, (r, c) = br_dyn(row, col)
    if st == 'cycle': return 0
    return row[2*r+c] if cote == 0 else col[2*r+c]
print()
print("Convention : gain au sorti de BR ; jeu injouable -> 0 pour chacun (pire que tout rang >= 1)")

Jeu sans NE pur : Ligne (2, 3, 1, 4) Colonne (1, 2, 4, 3)
Trajectoire BR depuis TG : TG -> TD -> BD -> BG -> TG -> TD -> ... (cycle)
Boucle : TD -> BD -> BG -> TG -> ... repetee indefiniment

Convention : gain au sorti de BR ; jeu injouable -> 0 pour chacun (pire que tout rang >= 1)


## 2. Marche 2 -- l'agent deplace le jeu, et paie

L'agent Ligne recoit desormais des **meta-actions** : appliquer l'un de ses trois swaps `R1, R2, R3` -- reecrire ses propres preference declarees -- puis jouer le jeu deplace. Chaque swap coute **c echelons de rang**. Cette unite n'est pas un artifice : payer 1 signifie *renoncer a un echelon de preference pour l'atteindre*, ce qui rend la soustraction `rang final - cout` honnetement interpretable dans un monde purement ordinal.

Ligne arbitre : rester et toucher son rang courant, ou migrer vers une table distante de d swaps et toucher `rang(final) - c*d`. Comme l'espace est fini, l'optimum se calcule par parcours exhaustif -- BFS depuis **sa propre table** (le graphe de Cayley des 24 tables, diametre 6, deja mesure par GT-3b).

In [4]:
# Marche 2 : optimum de migration par jeu, puis sweep du cout
def bfs_from(t0):
    d = {t0: 0}; q = deque([t0])
    while q:
        u = q.popleft()
        for k in (1, 2, 3):
            v = swap_val(u, k)
            if v not in d: d[v] = d[u] + 1; q.append(v)
    return d

best_by_dist = {}
for row_t, col_t in jeux:
    m = {}
    for t2, d in bfs_from(row_t).items():
        p = payoff(t2, col_t, 0)
        m[d] = max(m.get(d, -1), p)
    best_by_dist[(row_t, col_t)] = m   # m[0] = rang courant, m[d] = meilleur rang atteignable a distance d

print("c   | %migrer | dist.opt moy | gain net moy (migrants) | fuient un cycle")
print("-" * 72)
for c in (0, 1, 2, 3):
    n_mig = tot_d = tot_gain = fuit = 0
    for (r_t, c_t), m in best_by_dist.items():
        best = max(m[d] - c * d for d in m)
        bd = min(d for d in m if m[d] - c * d == best)
        if bd > 0 and best > m[0]:            # migrer ssi strictement meilleur que rester
            n_mig += 1; tot_d += bd; tot_gain += best - m[0]
            if payoff(r_t, c_t, 0) == 0: fuit += 1
    n = len(jeux)
    print(f"{c}   |  {100*n_mig//n:3d}%   |     {tot_d/max(n_mig,1):.2f}     |          {tot_gain/max(n_mig,1):+.2f}          |     {fuit}")

c   | %migrer | dist.opt moy | gain net moy (migrants) | fuient un cycle
------------------------------------------------------------------------
0   |   56%   |     1.33     |          +1.93          |     72
1   |   16%   |     1.25     |          +2.00          |     72
2   |    8%   |     1.00     |          +1.50          |     48
3   |    4%   |     1.00     |          +1.00          |     24


### Lecture du sweep : un seuil monotone, une distance qui se contracte

Quatre faits mesures :

1. **La migration s'effondre avec le cout** : 56 % des jeux voient Ligne migrer a cout nul, 16 % a cout 1, 8 % a cout 2, 4 % a cout 3. A un echelon par swap, six jeux sur sept restent -- le privilege de reecrire ses preferences est cher.
2. **La distance optimale se contracte** de 1,33 vers 1,00 : plus le km de swap est cher, plus on ne paie que le deplacement minimal -- a cout eleve, seuls les gains de proximite immediate survivent.
3. **Le gain net moyen des migrants monte puis redescend** (+1,93 a c=0, +2,00 a c=1, +1,50 a c=2, +1,00 a c=3) : a c=0 partent aussi les migrations gadgets ; a c=1 ne restent que les migrations de fond ; au-dela, le cout ronge le gain.
4. **Les 72 jeux injouables se vident a leur rythme** : meme a cout 1, les 72 fuient tous (quitter un gain nul vaut n'importe quel prix raisonnable) ; a cout 2 il en reste 48, a cout 3 seulement 24 -- les autres preferent le cycle gratuit au deplacement trop cher. *Meme l'injouable a un prix au-dessus duquel on le tolere.*

In [5]:
# Le Dilemme sous les meta-actions : que peut acheter Ligne ?
m_pd = best_by_dist[PD]
print("PD : rang courant de Ligne (defection mutuelle) =", m_pd[0])
print("Meilleur rang atteignable par distance :")
for d in sorted(m_pd):
    print(f"  a distance {d} : rang {m_pd[d]}" + (f"  -> net a c=1 : {m_pd[d]-d}" if d > 0 else "   (rester)"))
print()
for c in (0, 1, 2):
    best = max(m_pd[d] - c * d for d in m_pd)
    bd = min(d for d in m_pd if m_pd[d] - c * d == best)
    verdict = "RESTE" if bd == 0 else f"MIGRE a distance {bd}"
    print(f"c = {c} : net optimal {best} (courant {m_pd[0]}) -> Ligne {verdict}")

PD : rang courant de Ligne (defection mutuelle) = 2
Meilleur rang atteignable par distance :
  a distance 0 : rang 2   (rester)
  a distance 1 : rang 3  -> net a c=1 : 2
  a distance 2 : rang 4  -> net a c=1 : 2
  a distance 3 : rang 4  -> net a c=1 : 1
  a distance 4 : rang 4  -> net a c=1 : 0
  a distance 5 : rang 4  -> net a c=1 : -1
  a distance 6 : rang 4  -> net a c=1 : -2

c = 0 : net optimal 4 (courant 2) -> Ligne MIGRE a distance 2
c = 1 : net optimal 2 (courant 2) -> Ligne RESTE
c = 2 : net optimal 2 (courant 2) -> Ligne RESTE


### Lecture : le Dilemme exactement indifferent

A cout nul, Ligne quitte le Dilemme : deux swaps l'amennent a un jeu ou sa table lui donne le rang 4, net +2. Mais **a cout 1, l'indifference est exacte** : le rang 3 a distance 1 et le rang 4 a distance 2 donnent tous deux un net de 2 -- strictement egal a rester. Le gain de fuite du Dilemme est **entierement mange par le cout du deplacement**. Ce n'est pas un accident d'arrondi : sur l'echelle des rangs, la fuite solitaire du Dilemme coute precisement ce qu'elle rapporte. Le piege de la strate 6 a une propriete economique de plus : il est *juste assez solide* pour retenir un agent solitaire qui doit payer son deplacement -- ce qui rend d'autant plus interessante la question des deux agents, en marche 3.

In [6]:
# Les migrants a cout 1 : qui part, et la fuite des injouables
mig = []
for (r_t, c_t), m in best_by_dist.items():
    best = max(m[d] - d for d in m)
    bd = min(d for d in m if m[d] - d == best)
    if bd > 0 and best > m[0]:
        mig.append((r_t, c_t, bd, m[0], m[bd]))
print("Migrants a c=1 :", len(mig), "jeux sur", len(jeux), "dont", sum(1 for x in mig if x[3] == 0), "refugies (partent d'un rang nul)")
opp = [x for x in mig if x[3] > 0]
r_t, c_t, bd, p0, p1 = opp[0]
t_cible = [t for t, d in bfs_from(r_t).items() if d == bd and payoff(t, c_t, 0) == p1][0]
print("Opportuniste : jeu", (r_t, c_t), "| rang", p0, "-> rang", p1, "en", bd, "swap (table cible", t_cible, ")")

inj = [g for g in jeux if not ne_purs(*g)]
rep1 = sum(1 for g in inj if min(d for d in best_by_dist[g] if best_by_dist[g][d] > 0) == 1)
print()
print("Jeux injouables :", len(inj), "| reparables en 1 swap :", rep1, "| autres :", len(inj) - rep1)
row_i, col_i = inj[0]
jouables_d1 = [t for t, d in bfs_from(row_i).items() if d == 1 and payoff(t, col_i, 0) > 0]
meilleur = max(payoff(t, col_i, 0) for t in jouables_d1) if jouables_d1 else None
print("Exemple", (row_i, col_i), "-> tables jouables a distance 1 :", jouables_d1,
      "| meilleur rang", meilleur)

Migrants a c=1 : 96 jeux sur 576 dont 72 refugies (partent d'un rang nul)
Opportuniste : jeu ((1, 4, 2, 3), (1, 2, 4, 3)) | rang 2 -> rang 4 en 1 swap (table cible (2, 4, 1, 3) )

Jeux injouables : 72 | reparables en 1 swap : 48 | autres : 24
Exemple ((1, 3, 4, 2), (2, 1, 3, 4)) -> tables jouables a distance 1 : [(1, 2, 4, 3)] | meilleur rang 3


### Lecture : deux raisons de partir

Les migrants a cout 1 ne sont pas une population homogene. Il y a les **opportunistes** -- des jeux jouables ou un seul swap proche decalle l'equilibre vers une case de meilleur rang (l'exemple affiche saute de deux echelons en un swap) -- et les **refugies** : les 72 jeux sans equilibre, dont 48 se reparent a distance 1. Pour un refugie, la meta-action n'est pas du luxe strategique, c'est la condition meme de jouabilite : sans elle, le jeu ne produit aucun resultat -- c'est le refugie affiche plus bas, parti du rang nul. La strate 7 commence deja ici -- *changer les regles pour que le jeu existe* -- avant meme toute consideration d'amelioration.

### Le mur traversé — ce que le migrant coupe (critère de clôture #12207)

Le chantier #12207 se clôt quand un lecteur peut, dans un notebook exécuté : partir d'un jeu nommé, atteindre un autre jeu nommé par un chemin qu'il n'a pas écrit lui-même, **voir le mur qu'il traverse**, et lire le coût de la méta-action qui l'y a mené. Les marches 1-2 ont nommé les jeux, calculé les chemins optimaux et balayé les coûts — mais aucune sortie ne **montrait** le mur. Entre deux chambres strictes adjacentes, l'égalité des deux cases permutées est un mur de codimension 1 : la lecture Bruns-Kimmich de GT-3b, rendue visible ici dans le flot de migration payée.

In [7]:
# Le mur traverse : la meta-action echange les rangs k et k+1 de deux cases. Dans l'espace
# des utilites (representant lineaire u = valeur du rang), la deformation
# u(lambda) = (1-lambda)*u_depart + lambda*u_cible coupe l'hyperplan d'egalite des deux
# cases permutees. De part et d'autre, la table ordinale est CONSTANTE : c'est la chambre.
positions = [i for i in range(4) if r_t[i] != t_cible[i]]
assert len(positions) == 2, "une meta-action = un swap de deux cases"
iA, iB = positions
noms = ["TG", "TD", "BG", "BD"]

def table_depuis_u(u):
    """La table ordinale (rangs 1..4) induite par un profile d'utilites."""
    ordre = sorted(range(4), key=lambda j: u[j])
    t = [0] * 4
    for rang, j in enumerate(ordre, start=1):
        t[j] = rang
    return tuple(t)

print(f"Jeu depart (Ligne) : {r_t}  ->  cible : {t_cible}  ({bd} meta-action, rang {p0} -> {p1})")
print(f"Cases permutees : {noms[iA]} (rang {r_t[iA]}) <-> {noms[iB]} (rang {r_t[iB]})")
print()
print("lambda |   u_TG    u_TD    u_BG    u_BD  | table ordinale induite")
print("-" * 66)
for lam in (0.0, 0.25, 0.5, 0.75, 1.0):
    u = tuple((1 - lam) * r_t[j] + lam * t_cible[j] for j in range(4))
    if abs(u[iA] - u[iB]) < 1e-12:
        classe = f"MUR : u_{noms[iA]} = u_{noms[iB]} -- codimension 1"
    else:
        t = table_depuis_u(u)
        classe = "chambre de depart" if t == r_t else ("chambre cible" if t == t_cible else str(t))
    print(f"  {lam:.2f} | " + "  ".join(f"{x:5.2f}" for x in u) + f" | {classe}")
print()
print(f"Le migrant coupe UN mur (l'egalite {noms[iA]}/{noms[iB]} des rangs {min(r_t[iA], r_t[iB])} et {min(r_t[iA], r_t[iB]) + 1})")
print(f"a lambda = 0.50, et l'a paye c = {bd} : rang {p0} -> rang {p1}, gain net {p1 - p0 - bd:+d}.")

Jeu depart (Ligne) : (1, 4, 2, 3)  ->  cible : (2, 4, 1, 3)  (1 meta-action, rang 2 -> 4)
Cases permutees : TG (rang 1) <-> BG (rang 2)

lambda |   u_TG    u_TD    u_BG    u_BD  | table ordinale induite
------------------------------------------------------------------
  0.00 |  1.00   4.00   2.00   3.00 | chambre de depart
  0.25 |  1.25   4.00   1.75   3.00 | chambre de depart
  0.50 |  1.50   4.00   1.50   3.00 | MUR : u_TG = u_BG -- codimension 1
  0.75 |  1.75   4.00   1.25   3.00 | chambre cible
  1.00 |  2.00   4.00   1.00   3.00 | chambre cible

Le migrant coupe UN mur (l'egalite TG/BG des rangs 1 et 2)
a lambda = 0.50, et l'a paye c = 1 : rang 2 -> rang 4, gain net +1.


### Lecture : les quatre maillons enchaînés

La sortie ci-dessus est le critère de clôture de #12207 réalisé en un seul flot exécuté : un **jeu nommé** (le premier migrant opportuniste, tables affichées), rejoint par un **chemin qu'il n'a pas écrit** (le BFS optimal de la marche 2), en **traversant un mur visible** — la table ordinale reste celle de la chambre de départ pour tout λ < 0.5, celle de la chambre cible pour tout λ > 0.5, et l'égalité des deux cases permutées à λ = 0.50 exactement est le mur de codimension 1 — et le **coût lu** : c = 1 pour cette traversée, gain net +1 en rang.

Le passage payé se lit désormais `chambre -> mur -> chambre voisine` : la méta-action tarifée achète exactement une traversée de mur, et rien d'autre — un échange de rangs qui ne modifie l'ordre d'aucune autre case ne peut couper plus d'un mur.

## 3. Marche 3 -- deux agents, un meta-jeu

Derniere marche : **Ligne et Colonne disposent simultanement de leurs meta-actions**. Chacun choisit une action dans {rester, R1, R2, R3} x {rester, C1, C2, C3} -- chacun ne reecrit que **sa** table, payer ses propres echelons -- puis le jeu deplace est joue (dynamique BR depuis haut-gauche, convention de marche 1), et chacun touche son rang final moins son cout. Seize profils par jeu de base : un **meta-jeu 4x4** dont on peut chercher les equilibres de Nash purs, exactement comme a la marche 1.

La question du desaccord devient calculable : *existe-t-il un equilibre du meta-jeu, bouge-t-on a l'equilibre, et cet equilibre est-il bon pour les deux ?*

In [8]:
# Marche 3 : le meta-jeu 4x4 et ses equilibres (cout = 1)
def meta_profil(row_t, col_t, a1, a2, cout=1):
    r2 = swap_val(row_t, a1) if a1 > 0 else row_t
    c2 = swap_val(col_t, a2) if a2 > 0 else col_t
    p1 = payoff(r2, c2, 0); p2 = payoff(r2, c2, 1)
    return p1 - (cout if a1 > 0 else 0), p2 - (cout if a2 > 0 else 0)

def meta_ne(row_t, col_t, cout=1):
    profils = {(a1, a2): meta_profil(row_t, col_t, a1, a2, cout)
               for a1 in range(4) for a2 in range(4)}
    nes = []
    for (a1, a2), (u1, u2) in profils.items():
        if all(profils[(a1b, a2)][0] <= u1 for a1b in range(4)) and \
           all(profils[(a1, a2b)][1] <= u2 for a2b in range(4)):
            nes.append((a1, a2))
    return nes, profils

st_ne = Counter()
tous_restant = ambigu = aucun_restant = 0
for row_t, col_t in jeux:
    nes, _ = meta_ne(row_t, col_t)
    st_ne[len(nes)] += 1
    if not nes: continue
    if all(n == (0, 0) for n in nes): tous_restant += 1
    elif (0, 0) in nes: ambigu += 1
    else: aucun_restant += 1
print("Jeux avec au moins un meta-NE pur :", sum(v for k, v in st_ne.items() if k > 0), "/", len(jeux))
print("Distribution du nombre de meta-NE :", dict(sorted(st_ne.items())))
print()
print("Statut du mouvement a l'equilibre :")
print("  tous les meta-NE restent sur place :", tous_restant)
print("  (rester) coexiste avec des NE mobiles :", ambigu, "(selection d'equilibre)")
print("  aucun meta-NE ne reste sur place :", aucun_restant, "(bouger est necessaire)")

Jeux avec au moins un meta-NE pur : 572 / 576
Distribution du nombre de meta-NE : {0: 4, 1: 166, 2: 266, 3: 16, 4: 116, 5: 6, 6: 2}

Statut du mouvement a l'equilibre :
  tous les meta-NE restent sur place : 134
  (rester) coexiste avec des NE mobiles : 332 (selection d'equilibre)
  aucun meta-NE ne reste sur place : 106 (bouger est necessaire)


### Lecture : le meta-jeu a presque toujours des equilibres

Premiere surprise : **572 jeux sur 576** ont au moins un equilibre pur au niveau meta -- ajouter des meta-actions *stabilise* plutot qu'elle ne destabilise. Le detail du statut est plus riche :

- **134 jeux** : tous les equilibres restent sur place -- la meta-action existe mais personne n'a interet a s'en servir a l'equilibre (l'immobilisme est stable) ;
- **332 jeux** : rester et bouger coexistent comme equilibres -- c'est le domaine de la *selection d'equilibre* : plusieurs futurs possibles, aucun critere ordinal n'en designe un ;
- **106 jeux** : aucun equilibre sur place -- **bouger est une necessite d'equilibre**. Dans pres d'un jeu sur cinq, l'equilibre de strate 6 n'est plus tenable des que reecrire les regles est possible : l'agent qui refuse de payer reste piege pendant que l'autre reconfigure.

La marche 2 disait *quand un agent solitaire part* ; la marche 3 dit que **des qu'on ouvre la porte aux deux, dans un cinquieme des jeux, partir est force**.

In [9]:
# Le meta-jeu du Dilemme, en entier
nes_pd, prof_pd = meta_ne(*PD)
noms = ["reste", "R1", "R2", "R3"]
noms_c = ["reste", "C1", "C2", "C3"]
print("Meta-jeu du PD (Ligne en lignes, Colonne en colonnes) :")
print("          " + "".join(f"{n:>10s}" for n in noms_c))
for a1 in range(4):
    ligne = f"{noms[a1]:>8s}  "
    for a2 in range(4):
        u1, u2 = prof_pd[(a1, a2)]
        marque = " *" if (a1, a2) in nes_pd else "  "
        ligne += f"({u1},{u2}){marque} "
    print(ligne)
print()
print("* = equilibre de Nash pur du meta-jeu :", nes_pd)

Meta-jeu du PD (Ligne en lignes, Colonne en colonnes) :
               reste        C1        C2        C3
   reste  (2,2) * (4,1)   (2,2) * (2,1)   
      R1  (1,4)   (3,1)   (1,3)   (-1,-1)   
      R2  (2,2) * (3,1)   (2,2) * (2,1)   
      R3  (1,2)   (-1,-1)   (1,2)   (3,3) * 

* = equilibre de Nash pur du meta-jeu : [(0, 0), (0, 2), (2, 0), (2, 2), (3, 3)]


### Lecture : le Dilemme se resout -- a deux, et par equilibre

Le meta-jeu du Dilemme porte **cinq equilibres**. Quatre laissent les deux prisonniers a (2,2) -- le statu quo auquel on pense toujours. Mais le cinquieme, **(R3, C3), paie (3,3)** : chacun reecrit sa propre preference au niveau 3-4, le jeu deplace devient une coordination dont l'equilibre selectionne par la dynamique est la case mutuellement meilleure (rang 4 chacun), et meme apres avoir paye un echelon chacun, il reste **(3,3) > (2,2)**. Trois proprietes remarquables :

1. **L'evasion est unilateralement stable** : si un seul migre, la dynamique retombe sur la defection mutuelle et le migrant a paye pour rien -- les profils (R3, reste) et (reste, C3) du tableau paient (1, 2) et (2, 1) : moins que (3, 3) pour chacun -- personne ne peut profiter du départ de l'autre ;
2. **Elle est symetrique** : chacun ne touche qu'a sa table ; personne ne reecrit les preferences de l'autre ;
3. **Elle n'est pas unique** : quatre autres equilibres restent au (2,2). L'evasion *existe et est stable*, mais rien ne la *selectionne* -- c'est un probleme de focalisation, plus de stabilite.

C'est la reponse mesuree a la question de la marche : *modifier les regles est une action, et dans le Dilemme cette action est un equilibre qui domine le piege -- a condition que les deux la jouent ensemble.*

In [10]:
# L'echec de coordination : quand AUCUN equilibre n'est Pareto-optimal
def pareto_dominated(profils, p):
    return any(v[0] > p[0] and v[1] > p[1] for v in profils.values())

au_moins_un_bon = aucun_bon = 0
ex_dur = None
for row_t, col_t in jeux:
    nes, profils = meta_ne(row_t, col_t)
    if not nes: continue
    if any(not pareto_dominated(profils, profils[n]) for n in nes):
        au_moins_un_bon += 1
    else:
        aucun_bon += 1
        if ex_dur is None: ex_dur = (row_t, col_t, nes, profils)
print("Jeux (avec meta-NE) dont AU MOINS UN equilibre est Pareto-optimal :", au_moins_un_bon)
print("Jeux ou AUCUN equilibre n'est Pareto-optimal :", aucun_bon, "-- l'echec de coordination dur")
row_t, col_t, nes, profils = ex_dur
print()
print("Exemple", (row_t, col_t), ": les meta-NE paient", [profils[n] for n in nes])
frontiere = {k: v for k, v in profils.items() if not pareto_dominated(profils, v)}
print("Frontiere de Pareto du meta-jeu :", list(frontiere.values())[:5], "...")

Jeux (avec meta-NE) dont AU MOINS UN equilibre est Pareto-optimal : 568
Jeux ou AUCUN equilibre n'est Pareto-optimal : 4 -- l'echec de coordination dur

Exemple ((1, 3, 2, 4), (3, 4, 2, 1)) : les meta-NE paient [(2, 2), (2, 2), (2, 2), (2, 2)]
Frontiere de Pareto du meta-jeu : [(4, 1), (3, 1), (1, 3), (3, 1), (3, 3)] ...


### Lecture : la limite du decentralise

Dans **568 jeux sur 572**, au moins un equilibre du meta-jeu est Pareto-optimal : en general, le meilleur a deux est stable, le probleme est seulement de *le choisir* parmi les equilibres. Mais **4 jeux realisent l'echec de coordination pur** : tous leurs equilibres paient (2,2) tandis que la frontiere de Pareto contient des profils mutuellement meilleurs -- des regles reecrites dont les deux joueurs profiteraient, et qui ne sont **pas stables** : chacun serait tente d'en devier. L'exhibe le montre chiffres en main : l'accord existe dans la matrice, il n'existe pas dans les incitations.

C'est exactement la jonction annoncee par le chantier : la ou la strate 7 rencontre la theorie des mecanismes. Quand changer les regles est une action decentralisee, l'accord mutuel sur le changement reste un equilibre a produire -- le probleme ne fait que remonter d'un etage. Un protocole d'engagement (le versant commitment de [GT-9b](GameTheory-09b-Commitment-Stackelberg.ipynb)) est le candidat naturel, et l'exercice 3 le fait toucher du doigt.

## Limites et conventions (lues avant d'etendre)

- **Le cout en echelons de rang** est une convention locale, documentee : elle rend `rang - cout` interpretable sans cardinaliser les preferences. Un cout en energie, en temps ou en argent exigerait une echelle cardinale et changerait les seuils -- pas la forme du protocole.
- **La regle de jeu** est la dynamique BR depuis la case haut-gauche. Une autre regle de selection (depuis une autre case, tirage aleatoire, dynamique simultanee) deplace certains equilibres selectionnes ; les comptages d'equilibres *purs* (marche 1) et de meta-NE (marche 3) n'en dependent pas.
- **Le meta-jeu est one-shot et simultane** : pas de repetition, pas d'engagement, pas d'ordre de passage. L'exercice 3 ouvre le sequentiel.
- **Chacun ne reecrit que sa table** : le redesign institutionnel croise (reecrire la table de l'autre, ou une table commune) est hors perimetre -- c'est une strate encore au-dessus.
- Les dettes de verification du chantier (passage 576 vers 144, tore a 37 trous, references arXiv non ouvertes firsthand) restent **RAPPORTEES** et ne sont pas utilisees ici : ce notebook ne derive que du substrat exact de GT-3b, lui-meme exhaustivement verifie.

## Exercice 1 -- le sweep vu par Colonne

La marche 2 a fait migrer Ligne. Refaites le meme sweep du cout **cote Colonne** (ses swaps C1-C3, son gain au sorti de la dynamique). Le compte de migrants a c=1 est-il le meme que cote Ligne (16 %) ? Si oui, dites pourquoi la symetrie l'impose ; sinon, exhibez un jeu qui les distingue.

In [11]:
# Exercice 1 : sweep du cout cote Colonne
# TODO etudiant : reconstruire best_by_dist_col (BFS sur la table Colonne, payoff cote 1)
# puis le tableau c=0..3 : %migrer, distance optimale moyenne, fuient-un-cycle
resultat_ex1 = None  # TODO etudiant : {c: (pct_migrer, dist_moyenne, fuites)}
print("Exercice 1 : a completer (sweep cote Colonne)")

Exercice 1 : a completer (sweep cote Colonne)


## Exercice 2 -- deplacer la selection d'equilibre

Les 72 jeux a deux equilibres purs sont les coordinations. En partant de l'un d'eux, montrez un cas ou **un seul swap de Ligne change l'equilibre selectionne** par la dynamique (la case d'arrivee change), et un cas ou aucun swap ne la change. Combien des 72 jeux sont du premier type ? (Le comptage exhaustif est possible : 72 jeux x 24 tables.)

In [12]:
# Exercice 2 : la selection d'equilibre sous meta-actions
# TODO etudiant : pour chaque jeu a 2 NE purs, comparer l'arrivee de br_dyn pour la table
# courante et pour les 23 autres tables ; compter ceux ou l'arrivee peut changer
resultat_ex2 = None  # TODO etudiant : (nb_jeux_selection_deplacable, exemple)
print("Exercice 2 : a completer (selection deplacable)")

Exercice 2 : a completer (selection deplacable)


## Exercice 3 -- le meta-jeu sequentiel

La marche 3 est simultanee : les 4 echecs de coordination dur proviennent de l'instabilite de l'accord. Rendez le meta-jeu **sequentiel** : Ligne annonce et joue sa meta-action, Colonne voit le jeu deplace puis choisit la sienne. Proposez le protocole (qui observe quoi, ordre des couts), recalculez les equilibres (backward induction sur les 4x4 sous-jeux), et dites combien des 4 echecs durs se referment. Verdict attendu en une phrase : *le sequentiel suffit-il a stabiliser l'accord ?*

In [13]:
# Exercice 3 : equilibres parfaits du meta-jeu sequentiel
# TODO etudiant : pour chacun des 4 jeux en echec dur, construire l'arbre
# (Ligne : 4 actions ; Colonne observe ; 4 actions) et resoudre par induction
resultat_ex3 = None  # TODO etudiant : {jeu: equilibre_sequentiel, referme: bool}
print("Exercice 3 : a completer (meta-jeu sequentiel)")

Exercice 3 : a completer (meta-jeu sequentiel)


## Conclusion : trois marches, un franchissement

Les trois marches mesurent le passage de la strate 6 a la strate 7 sur un univers fini ou tout se verifie :

| Marche | Ce que fait l'agent | Chiffre cle |
|---|---|---|
| 1 -- jouer | subit les regles, converge ou cycle | 72 jeux injouables sur 576 ; convergence si et seulement si equilibre pur |
| 2 -- deplacer | paie des echelons pour reecrire sa table | 56 % -> 16 % -> 8 % -> 4 % de migrants quand le cout monte ; le Dilemme exactement indifferent a c=1 |
| 3 -- se coordonner sur les regles | joue le meta-jeu 4x4 | 106 jeux ou bouger est necessaire ; le Dilemme s'evade a (3,3) par equilibre symetrique ; 4 echecs de coordination dur |

Le resultat qui donne son sens au chantier : **changer les regles est une action ordinaire** -- elle a un prix, un seuil, des equilibres, et meme ses propres echecs de coordination. Le vocabulaire strategique de la strate 6 s'applique sans modification un etage plus haut ; c'est la definition meme d'un franchissement reussi.

*Suite du chantier (#12207)* : D1 (la grammaire des generateurs manipulables) et D3 (le chemin minimal certifie, cible Lean a terme via #12205) restent a ecrire sur le meme substrat.

***

[← GameTheory-03-Topology2x2](GameTheory-03-Topology2x2.ipynb) | [GameTheory-03b-Chambres-et-Murs →](GameTheory-03b-Chambres-et-Murs.ipynb) | [↑ README GameTheory](README.md)

*GameTheory 3e -- Meta-Actions Tarifees, versant D4 du chantier « Les jeux comme objets » (#12207). Substrat, encodages et swaps herites de GT-3b ; jumeaux conceptuels GT-21 (les deux especes de fleches) et GT-20 (l'engagement comme protocole de coordination sur les regles).*